# Chapter 7. Data Cleaning and Preparation

In [1]:
import numpy as np
import pandas as pd

## 7.2 Data Transformation
### Removing Duplicates

In [2]:
data = pd.DataFrame({'k1':['one','two']*3 + ['two'],
                    'k2':[1,1,2,3,3,4,4]})
data

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


> - The method ***duplicated*** **returns a boolean Series** indicating whether each row is a duplicate or not:
> - drop_duplicates **returns a DataFrame where the duplicated array is False**:

In [3]:
data.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

In [4]:
data.drop_duplicates()

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


> - By default consider all of the columns; alternatively, you can specify any subset of them to detect duplicates.
> - Passing ***keep = 'last'*** will return the last one:

In [5]:
data['v1'] = range(7)
data.drop_duplicates(['k1'])

,k1,k2,v1
0,one,1,0
1,two,1,1


In [6]:
data.drop_duplicates(['k1', 'k2'], keep='last')

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
6,two,4,6


### Transforming Data Using a Function or Mapping

In [7]:
data = pd.DataFrame({'food':['bacon','pulled pork', 'bacon',
                            'Pastrami','corned beef','Bacon',
                            'pastrami','honey ham','nova lox'],
                    'ounces':[4,3,12,6,7.5,8,3,5,6]})
data

,food,ounces
0,bacon,4.0
1,pulled pork,3.0
2,bacon,12.0
3,Pastrami,6.0
4,corned beef,7.5
5,Bacon,8.0
6,pastrami,3.0
7,honey ham,5.0
8,nova lox,6.0


In [8]:
meat_to_animal = {'bacon':'pig','pulled pork':'pig',
                  'pastrami':'cow', 'corned beef':'cow',
                 'honey ham':'pig', 'nova lox':'salmon'}

> - The ***map* method on a Series** accepts a function or dict-like object containing a mapping
> - Using ***map*** is a convenient way to perform element-wise transformations and other data cleaning-related operations.

In [10]:
lowercased = data['food'].str.lower()
lowercased

0          bacon
1    pulled pork
2          bacon
3       pastrami
4    corned beef
5          bacon
6       pastrami
7      honey ham
8       nova lox
Name: food, dtype: object

In [11]:
data['animal'] = lowercased.map(meat_to_animal)
data

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,Pastrami,6.0,cow
4,corned beef,7.5,cow
5,Bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


In [12]:
data['food'].map(lambda x: meat_to_animal[x.lower()])
# meat_to_animal['Pastrami'.lower()] -> 'cow'

0       pig
1       pig
2       pig
3       cow
4       cow
5       pig
6       cow
7       pig
8    salmon
Name: food, dtype: object

### Replacing Values

In [2]:
data = pd.Series([1.,-999,2.,-999,-1000,3.])
data

0       1.0
1    -999.0
2       2.0
3    -999.0
4   -1000.0
5       3.0
dtype: float64

In [3]:
data.replace([-999, -1000], np.nan) # return new object, 원 데이터 변화 x

0    1.0
1    NaN
2    2.0
3    NaN
4    NaN
5    3.0
dtype: float64

> - To use a different replacement for each value, pass a list of substitutes:

In [4]:
data.replace([-999,-1000], [np.nan, 0])
# = data.replace({-999: np.nan, -1000: 0}) dict type

0    1.0
1    NaN
2    2.0
3    NaN
4    0.0
5    3.0
dtype: float64

### Renaming Axis Indexes
- Like values in a Series, **axis labels** can be similarly transformed by a function or mapping of some form to produce new, differently labeled objects.

In [5]:
data = pd.DataFrame(np.arange(12).reshape((3,4)), 
                    index=['Ohio','Colorado','New York'],
                   columns=['one','two','three','four'])

> - Like a Series, the axis indexes have a **map method**:
> - You can assign to index, modifying the DataFrame in-place:

In [8]:
transform = lambda x: x[:4].upper()
data.index.map(transform)

Index(['OHIO', 'COLO', 'NEW '], dtype='object')

In [11]:
data.index = data.index.map(transform) # 원 DataFrame 변경
data

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


> - Create a transformed version of a dataset without modifying the original : ***rename***
> - conjunction with a dict-like object providing new values for a subset of the axis labels:

In [15]:
data.rename(index=str.title, columns=str.upper)

,ONE,TWO,THREE,FOUR
Ohio,0,1,2,3
Colo,4,5,6,7
New,8,9,10,11


In [16]:
data.rename(index={'OHIO':'INDIANA'}, columns={'three':'peekaboo'})

,one,two,peekaboo,four
INDIANA,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


### Discretization and Binning
- Continuous data is often discretized or otherwise seperated into "bins" for analysis.

In [2]:
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]
bins = [18, 25, 35, 60, 100]

In [3]:
cats = pd.cut(ages, bins)
cats

[(18, 25], (18, 25], (18, 25], (25, 35], (18, 25], ..., (25, 35], (60, 100], (35, 60], (35, 60], (25, 35]]
Length: 12
Categories (4, interval[int64, right]): [(18, 25] < (25, 35] < (35, 60] < (60, 100]]

> - **Categorical object** : You can treat it like an array of strings indicating the bin name

In [4]:
cats.codes

array([0, 0, 0, 1, 0, 0, 2, 1, 3, 2, 2, 1], dtype=int8)

In [5]:
cats.categories

IntervalIndex([(18, 25], (25, 35], (35, 60], (60, 100]], dtype='interval[int64, right]')

> - Consistent with mathmatical notation for intervals, a **parenthesis** means that the side is ***open***, while the **square bracket** means it is ***closed(inclusive)***.

In [4]:
pd.cut(ages, [18, 26, 36, 61, 100], right=False)

[[18, 26), [18, 26), [18, 26), [26, 36), [18, 26), ..., [26, 36), [61, 100), [36, 61), [36, 61), [26, 36)]
Length: 12
Categories (4, interval[int64, left]): [[18, 26) < [26, 36) < [36, 61) < [61, 100)]

> - Pass your own **bin names** by passing a list or array to the ***labels*** option:

In [11]:
group_name = ['Youth','YoungAdult','MiddleAged','Senior']
pd.cut(ages, bins, labels=group_name)

['Youth', 'Youth', 'Youth', 'YoungAdult', 'Youth', ..., 'YoungAdult', 'Senior', 'MiddleAged', 'MiddleAged', 'YoungAdult']
Length: 12
Categories (4, object): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']

> - If you **pass an integer number of bins** to cut instead of explicit bin edge, it will compute equal-length bons based on the minimum and maximum values in the data

In [3]:
data = np.random.rand(20)
pd.cut(data, 4, precision=2) # limits the decimal precision

[(0.74, 0.98], (-0.00014, 0.25], (0.74, 0.98], (0.25, 0.49], (0.25, 0.49], ..., (-0.00014, 0.25], (0.49, 0.74], (0.25, 0.49], (-0.00014, 0.25], (0.25, 0.49]]
Length: 20
Categories (4, interval[float64, right]): [(-0.00014, 0.25] < (0.25, 0.49] < (0.49, 0.74] < (0.74, 0.98]]

> - A closely related functions, ***qcut***, bins the data based on sample **quantiles**.
> - Since ***qcut*** uses sample quantiles instead, by definition you will obtain roughly equal-size bins:

In [6]:
data = np.random.randn(1000) # nomally distributed
cats = pd.qcut(data, 4) # Cut into quartiles
cats

[(-3.0829999999999997, -0.681], (0.716, 2.886], (0.716, 2.886], (0.716, 2.886], (-0.0167, 0.716], ..., (0.716, 2.886], (-3.0829999999999997, -0.681], (0.716, 2.886], (-0.681, -0.0167], (-0.681, -0.0167]]
Length: 1000
Categories (4, interval[float64, right]): [(-3.0829999999999997, -0.681] < (-0.681, -0.0167] < (-0.0167, 0.716] < (0.716, 2.886]]

In [7]:
pd.value_counts(cats)

(-3.0829999999999997, -0.681]    250
(-0.681, -0.0167]                250
(-0.0167, 0.716]                 250
(0.716, 2.886]                   250
dtype: int64

> - Similar to cut you can pass your own quantiles (numbers between 0 and 1, inclusive)

In [8]:
pd.qcut(data, [0,0.1,0.5,0.9,1])

[(-1.266, -0.0167], (1.328, 2.886], (1.328, 2.886], (-0.0167, 1.328], (-0.0167, 1.328], ..., (-0.0167, 1.328], (-1.266, -0.0167], (-0.0167, 1.328], (-1.266, -0.0167], (-1.266, -0.0167]]
Length: 1000
Categories (4, interval[float64, right]): [(-3.0829999999999997, -1.266] < (-1.266, -0.0167] < (-0.0167, 1.328] < (1.328, 2.886]]